In [7]:
from pathlib import Path
import pandas as pd

In [8]:
OUT = Path("/.../data/output")
ACCOUNTS = ["REV", "COGS", "PAYROLL", "MARKETING", "IT", "TRAVEL"]

In [21]:
#Load finance data
def load_finance_data(version):
    fn = {"actual" : "fact_finance_actual.csv", "budget" : "fact_finance_budget.csv"}.get(version)
    if fn is None and version.startswith("forecast_") : fn = f"fact_finance_{version}.csv"
    if fn is None: raise ValueError(f"Unknown version: {version}")
    return pd.read_csv(OUT/fn, parse_dates = ["date"])

In [30]:
def calculate_pnl(finance):
    missing_columns = {"date", "region_id", "account_id", "value"} - set(finance.columns)
    if missing_columns: raise ValueError(f"Missing columns: {missing_columns}")
    
    finance_pivot = finance.pivot_table(index = ["date", "region_id"], columns = "account_id", values = "value", aggfunc = "sum", fill_value = 0).reset_index()
    for account in ACCOUNTS: 
        if account not in finance_pivot.columns: finance_pivot[account] = 0.0 #Missing account has value 0
    finance_pivot = finance_pivot.rename(columns = {"REV" : "revenue", "COGS" : "cogs", "PAYROLL" : "payroll", "MARKETING" : "marketing", "IT" : "it", "TRAVEL" : "travel"})
    
    finance_pivot["gross profit"] = finance_pivot["revenue"] - finance_pivot["cogs"]
    finance_pivot["gross margin"] = finance_pivot["gross profit"] / (finance_pivot["revenue"].replace(0, pd.NA))
    finance_pivot["opex"] = finance_pivot[["payroll", "marketing", "it", "travel"]].sum(axis=1)
    finance_pivot["ebitda"] = finance_pivot["gross profit"] + finance_pivot["opex"]
    finance_pivot["ebitda margin"] = finance_pivot["ebitda"] / (finance_pivot["revenue"].replace(0, pd.NA))
    
    return finance_pivot[["date", "region_id", "revenue", "cogs", "gross profit", "payroll", "marketing", "it", "travel", "opex", "ebitda", "ebitda margin"]]


In [31]:
VERSIONS=['actual','budget','forecast_v1','forecast_v2','forecast_v3']

In [32]:
def build_all_pnl():
    frames = []
    for version in VERSIONS:
        p = calculate_pnl(load_finance_data(version))
        p["version"] = version
        p.to_csv(OUT/f"pnl_{version}.csv", index = False)
        frames.append(p)
        c = pd.concat(frames, ignore_index = True)
        c.to_csv(OUT/f"pnl_all_versions.csv", index = False)
    return c

In [33]:
build_all_pnl()

account_id,date,region_id,revenue,cogs,gross profit,payroll,marketing,it,travel,opex,ebitda,ebitda margin,version
0,2024-01-01,APAC,645985.085743,-303612.990299,9.495981e+05,-70833.333333,-18333.333333,-9166.666667,-6250.000,-104583.333333,8.450147e+05,1.308103,actual
1,2024-01-01,EU,761369.223512,-357843.535051,1.119213e+06,-70833.333333,-18333.333333,-9166.666667,-6250.000,-104583.333333,1.014629e+06,1.332638,actual
2,2024-01-01,OTHER,537595.821438,-252670.036076,7.902659e+05,-70833.333333,-18333.333333,-9166.666667,-6250.000,-104583.333333,6.856825e+05,1.275461,actual
3,2024-02-01,APAC,776667.833965,-365033.881964,1.141702e+06,-70833.333333,-18333.333333,-9166.666667,-6250.000,-104583.333333,1.037118e+06,1.335344,actual
4,2024-02-01,EU,809839.524321,-380624.576431,1.190464e+06,-70833.333333,-18333.333333,-9166.666667,-6250.000,-104583.333333,1.085881e+06,1.340859,actual
...,...,...,...,...,...,...,...,...,...,...,...,...,...
535,2026-11-01,EU,709069.170507,-328313.066918,1.037382e+06,-70302.083333,-18195.833333,-9097.916667,-6203.125,-103798.958333,9.335833e+05,1.316632,forecast_v3
536,2026-11-01,OTHER,797813.590461,-369403.490672,1.167217e+06,-70302.083333,-18195.833333,-9097.916667,-6203.125,-103798.958333,1.063418e+06,1.332916,forecast_v3
537,2026-12-01,APAC,907679.319260,-420273.498665,1.327953e+06,-70302.083333,-18195.833333,-9097.916667,-6203.125,-103798.958333,1.224154e+06,1.348663,forecast_v3
538,2026-12-01,EU,758863.223269,-351368.699368,1.110232e+06,-70302.083333,-18195.833333,-9097.916667,-6203.125,-103798.958333,1.006433e+06,1.326238,forecast_v3


In [47]:
def build_pnl_variance():
    joint_column = ["date", "region_id"]
    a = calculate_pnl(load_finance_data("actual"))
    a = a.rename(columns = {c:f"{c} actual" for c in a.columns if c not in joint_column})
    b = calculate_pnl(load_finance_data("budget"))
    b = b.rename(columns = {c:f"{c} budget" for c in b.columns if c not in joint_column})
    r = a.merge(b, on = joint_column, how = "outer")
    
    for m in ["revenue", "cogs", "gross profit", "payroll", "marketing", "it", "travel", "opex", "ebitda", "ebitda margin"]:
        r[f"{m} variance"] = r[f"{m} actual"] - r[f"{m} budget"]
        r[f"{m} variance pct"] = r[f"{m} variance"] / (r[f"{m} budget"].replace(0, pd.NA))
        r["ebitda margin variance pp"] = (r["ebitda margin actual"] - r["ebitda margin budget"])  * 100
    
    r.to_csv(OUT/f"pnl_actual_vs_budget.csv", index = False)
    return r

In [48]:
build_pnl_variance()

account_id,date,region_id,revenue actual,cogs actual,gross profit actual,payroll actual,marketing actual,it actual,travel actual,opex actual,...,it variance,it variance pct,travel variance,travel variance pct,opex variance,opex variance pct,ebitda variance,ebitda variance pct,ebitda margin variance,ebitda margin variance pct
0,2024-01-01,APAC,645985.085743,-303612.990299,9.495981e+05,-70833.333333,-18333.333333,-9166.666667,-6250,-104583.333333,...,-275.0,0.030928,-187.5,0.030928,-3137.5,0.030928,-22904.643624,-0.026390,0.016218,0.012554
1,2024-01-01,EU,761369.223512,-357843.535051,1.119213e+06,-70833.333333,-18333.333333,-9166.666667,-6250,-104583.333333,...,-275.0,0.030928,-187.5,0.030928,-3137.5,0.030928,-26435.398239,-0.025393,0.017870,0.013592
2,2024-01-01,OTHER,537595.821438,-252670.036076,7.902659e+05,-70833.333333,-18333.333333,-9166.666667,-6250,-104583.333333,...,-275.0,0.030928,-187.5,0.030928,-3137.5,0.030928,-19587.932136,-0.027774,0.014021,0.011115
3,2024-02-01,APAC,776667.833965,-365033.881964,1.141702e+06,-70833.333333,-18333.333333,-9166.666667,-6250,-104583.333333,...,-275.0,0.030928,-187.5,0.030928,-3137.5,0.030928,-26903.535719,-0.025285,0.018052,0.013704
4,2024-02-01,EU,809839.524321,-380624.576431,1.190464e+06,-70833.333333,-18333.333333,-9166.666667,-6250,-104583.333333,...,-275.0,0.030928,-187.5,0.030928,-3137.5,0.030928,-27918.589444,-0.025066,0.018423,0.013931
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,2026-11-01,EU,702048.683670,-329962.881325,1.032012e+06,-70833.333333,-18333.333333,-9166.666667,-6250,-104583.333333,...,-275.0,0.030928,-187.5,0.030928,-3137.5,0.030928,-24620.189720,-0.025860,0.017089,0.013105
104,2026-11-01,OTHER,789914.446001,-371259.789620,1.161174e+06,-70833.333333,-18333.333333,-9166.666667,-6250,-104583.333333,...,-275.0,0.030928,-187.5,0.030928,-3137.5,0.030928,-27308.882048,-0.025195,0.018204,0.013797
105,2026-12-01,APAC,898692.395307,-422385.425794,1.321078e+06,-70833.333333,-18333.333333,-9166.666667,-6250,-104583.333333,...,-275.0,0.030928,-187.5,0.030928,-3137.5,0.030928,-30637.487296,-0.024566,0.019283,0.014451
106,2026-12-01,EU,751349.726009,-353134.371224,1.104484e+06,-70833.333333,-18333.333333,-9166.666667,-6250,-104583.333333,...,-275.0,0.030928,-187.5,0.030928,-3137.5,0.030928,-26128.801616,-0.025466,0.017747,0.013515
